# 🚀 Forecasting Pipeline — Datathon VinTelligence 2026
**Revenue & COGS Daily Forecasting | 2023-01-01 → 2024-07-01 (548 days)**

## Pipeline Overview

| Notebook | Mô tả |
|----------|--------|
| `01_data_preparation` | Load, validate, clean, feature engineering (calendar, Tet, aux profiles) |
| `02_baseline_models`  | Prophet + LightGBM residual (v3: AutoSearch 17 profiles) |
| `03_advanced_ensemble`| v4 Top Kill, v5 Naive-First paradigm shift |
| `04_breakthrough`     | v10/v11: 10 strategies + scale correction + M-competition models |
| **`forecasting`** | **Tổng hợp toàn bộ pipeline, chạy end-to-end, tạo submission** |

## Key Insights
- ✅ **Naive364 (MAE ≈ 830k) beats Prophet alone (MAE ≈ 2.3M)** → Naive-first architecture
- ✅ **COGS/Revenue ratio rất ổn định** → dự đoán Revenue, derive COGS từ ratio
- ✅ **Post-2019 structural break** (COVID) → train từ 2019+ cho short-range
- ✅ **Tet empirical multipliers** từ dữ liệu lịch sử > generic window flags
- ✅ **Scale correction** → match prediction level với sample_submission


## 0. Environment Setup & Dependencies

In [ ]:
# ── Auto-install dependencies ──────────────────────────────────────
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

PKG_MAP = {
    'prophet':       'prophet',
    'lightgbm':      'lightgbm',
    'catboost':      'catboost',
    'statsforecast': 'statsforecast',
}
for import_name, pkg_name in PKG_MAP.items():
    try:
        __import__(import_name)
    except ImportError:
        print(f'Installing {pkg_name}...')
        _install(pkg_name)

print('✅ All dependencies ready.')


In [ ]:
import warnings
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from scipy import stats as scipy_stats
from scipy.signal import savgol_filter
from sklearn.linear_model import LinearRegression
from prophet import Prophet
import lightgbm as lgb

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Optional dependencies ──────────────────────────────────────────
try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print('⚠️  CatBoost not available — Strategy G will be skipped.')

try:
    from statsforecast import StatsForecast
    from statsforecast.models import AutoTheta, AutoETS, SeasonalNaive
    HAS_STATSFORECAST = True
except ImportError:
    HAS_STATSFORECAST = False
    print('⚠️  statsforecast not available — AutoTheta/AutoETS will be skipped.')

print(f'CatBoost     : {HAS_CATBOOST}')
print(f'StatsForecast: {HAS_STATSFORECAST}')


In [ ]:
# ── Auto-detect environment & set paths ────────────────────────────
def _detect_base_dir():
    """Works on Google Colab, local Windows, and local Linux."""
    if 'google.colab' in sys.modules or os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        return Path('/content/drive/My Drive/Datathon_VinTelligence')
    try:
        nb_dir = Path(__file__).resolve().parent
    except NameError:
        nb_dir = Path().resolve()
    for candidate in [nb_dir, nb_dir.parent, nb_dir.parent.parent]:
        if (candidate / 'data').exists():
            return candidate
    return nb_dir.parent

BASE_DIR   = _detect_base_dir()
DATA_DIR   = BASE_DIR / 'data' / 'datathon-2026-round-1'
CLEAN_DIR  = BASE_DIR / 'data' / 'data_clean'
OUTPUT_DIR = BASE_DIR / 'output'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Global constants ───────────────────────────────────────────────
FORECAST_START = pd.Timestamp('2023-01-01')
FORECAST_END   = pd.Timestamp('2024-07-01')
TRAIN_END      = pd.Timestamp('2022-12-31')
N_FORECAST     = 548
TARGET_COLS    = ['Revenue', 'COGS']

forecast_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')
assert len(forecast_dates) == N_FORECAST

TET_DATES = pd.to_datetime([
    '2012-01-23','2013-02-10','2014-01-31','2015-02-19',
    '2016-02-08','2017-01-28','2018-02-16','2019-02-05',
    '2020-01-25','2021-02-12','2022-02-01','2023-01-22','2024-02-10',
])
VN_HOLIDAYS   = [(1,1),(4,30),(5,1),(9,2)]
MEGA_SALE_DAYS= [(3,3),(4,4),(5,5),(6,6),(7,7),(8,8),(9,9),(10,10),(11,11),(12,12)]

plt.rcParams.update({
    'figure.figsize': (16, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print(f'ENV       : {"Colab" if os.path.exists("/content") else "Local"}')
print(f'BASE_DIR  : {BASE_DIR}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'CLEAN_DIR : {CLEAN_DIR}')
print(f'Forecast  : {FORECAST_START.date()} → {FORECAST_END.date()} ({N_FORECAST} days)')


## 1. Load & Validate Raw Data

In [ ]:
# ── Load raw datasets ─────────────────────────────────────────────
df_sales   = pd.read_csv(DATA_DIR / 'sales.csv',       parse_dates=['Date'])
df_web     = pd.read_csv(DATA_DIR / 'web_traffic.csv', parse_dates=['date'])
df_orders  = pd.read_csv(DATA_DIR / 'orders.csv',      parse_dates=['order_date'])
df_returns = pd.read_csv(DATA_DIR / 'returns.csv',     parse_dates=['return_date'])
df_promos  = pd.read_csv(DATA_DIR / 'promotions.csv',  parse_dates=['start_date','end_date'])
df_sample  = pd.read_csv(DATA_DIR / 'sample_submission.csv', parse_dates=['Date'])

df_sales = df_sales.sort_values('Date').reset_index(drop=True)

# ── Quick validation ──────────────────────────────────────────────
print('=== Dataset Overview ===')
for name, df in [('Sales', df_sales), ('Web', df_web), ('Orders', df_orders),
                  ('Returns', df_returns), ('Promos', df_promos)]:
    print(f'  {name:8s}: {len(df):,} rows | {df.shape[1]} cols | '
          f'missing {df.isnull().mean().mean()*100:.1f}%')

print(f'\nSales range: {df_sales.Date.min().date()} → {df_sales.Date.max().date()}')
print(f'Negative Revenue: {(df_sales.Revenue <= 0).sum()}')
print(f'Negative COGS   : {(df_sales.COGS <= 0).sum()}')

# Check for missing dates
full_range    = pd.date_range(df_sales.Date.min(), df_sales.Date.max(), freq='D')
missing_dates = full_range.difference(df_sales.Date)
print(f'Missing dates   : {len(missing_dates)}')

# YoY summary
df_sales['year'] = df_sales.Date.dt.year
yoy = df_sales.groupby('year')[['Revenue', 'COGS']].sum()
yoy['Gross_Margin%'] = ((yoy.Revenue - yoy.COGS) / yoy.Revenue * 100).round(2)
yoy['COGS_Ratio']    = (yoy.COGS / yoy.Revenue).round(4)
print('\nYear-over-Year Summary:')
display(yoy)


## 2. Vietnamese Calendar Feature Engineering

In [ ]:
def build_calendar_features(date_range: pd.DatetimeIndex) -> pd.DataFrame:
    """Build Vietnamese calendar features for a date range (zero leakage)."""
    df = pd.DataFrame({'ds': date_range})
    df['date']    = df.ds
    df['year']    = df.ds.dt.year
    df['month']   = df.ds.dt.month
    df['day']     = df.ds.dt.day
    df['dow']     = df.ds.dt.dayofweek   # 0=Mon, 6=Sun
    df['doy']     = df.ds.dt.dayofyear
    df['week']    = df.ds.dt.isocalendar().week.astype(int)
    df['quarter'] = df.ds.dt.quarter
    df['is_weekend']     = (df.dow >= 5).astype(int)
    df['is_month_start'] = (df.day == 1).astype(int)
    df['is_month_end']   = df.ds.dt.is_month_end.astype(int)
    df['is_payday']      = df.day.isin([1, 15, 25]).astype(int)
    df['is_quarter_end'] = df.ds.dt.is_quarter_end.astype(int)

    # Cyclic encoding
    df['month_sin'] = np.sin(2 * np.pi * df.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df.month / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df.dow / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df.dow / 7)

    # Vietnamese national holidays
    df['is_vn_holiday'] = df.apply(
        lambda r: int((r.month, r.day) in VN_HOLIDAYS), axis=1)

    # Mega sale days (11.11, 12.12, etc.) ± 3-day window
    df['is_mega_sale'] = df.apply(
        lambda r: int((r.month, r.day) in MEGA_SALE_DAYS), axis=1)

    mega_dates = []
    for yr in df.year.unique():
        for (m, d) in MEGA_SALE_DAYS:
            try: mega_dates.append(pd.Timestamp(yr, m, d))
            except: pass
    mega_dates = pd.DatetimeIndex(mega_dates)
    df['is_sale_window'] = df.ds.apply(
        lambda d: int(any(abs((d - md).days) <= 3 for md in mega_dates))).astype(int)

    # Tet proximity (days to nearest Tet, clipped to [-30, 30])
    def tet_dist(d):
        deltas = [(d - t).days for t in TET_DATES]
        closest = min(deltas, key=abs)
        return float(np.clip(closest, -30, 30))
    df['tet_days']      = df.ds.apply(tet_dist)
    df['tet_proximity'] = np.exp(-np.abs(df.tet_days) / 10)

    return df.set_index('ds').drop(columns=['date'])


# Build calendar for full training + forecast range
all_dates = pd.date_range(df_sales.Date.min(), FORECAST_END, freq='D')
df_cal    = build_calendar_features(all_dates)
print(f'Calendar features shape: {df_cal.shape}')
print(f'Columns: {df_cal.columns.tolist()}')


## 3. Prophet Holiday DataFrame

In [ ]:
def build_prophet_holidays() -> pd.DataFrame:
    """Build Prophet-compatible holiday DataFrame with pre/post windows."""
    rows = []
    for tet_dt in TET_DATES:
        rows.append(dict(holiday='tet', ds=tet_dt, lower_window=-21, upper_window=7))
    for yr in range(2012, 2025):
        for (m, d) in VN_HOLIDAYS:
            try:
                rows.append(dict(holiday='vn_holiday', ds=pd.Timestamp(yr,m,d),
                                 lower_window=-1, upper_window=1))
            except: pass
        for (m, d) in MEGA_SALE_DAYS:
            try:
                rows.append(dict(holiday='mega_sale', ds=pd.Timestamp(yr,m,d),
                                 lower_window=-3, upper_window=3))
            except: pass
    df_h = pd.DataFrame(rows)
    df_h['ds'] = pd.to_datetime(df_h['ds'])
    return df_h.drop_duplicates(subset=['holiday','ds'])

df_holidays = build_prophet_holidays()
print(f'Prophet holidays: {len(df_holidays)} events')
print(df_holidays.groupby('holiday').size())


## 4. Auxiliary Feature Profiles (Zero Leakage)

In [ ]:
# ── Web traffic profile — median by (month, DOW) ────────────────
df_web['month'] = df_web.date.dt.month
df_web['dow']   = df_web.date.dt.dayofweek
web_profile = (
    df_web.groupby(['month','dow'])[['sessions','unique_visitors','page_views']]
    .median().reset_index()
)
web_profile.columns = ['month','dow','med_sessions','med_visitors','med_pageviews']

# ── Order count profile ───────────────────────────────────────────
df_orders['month'] = df_orders.order_date.dt.month
df_orders['dow']   = df_orders.order_date.dt.dayofweek
order_daily = df_orders.groupby(['order_date','month','dow']).size().reset_index(name='order_count')
order_profile = (order_daily.groupby(['month','dow'])['order_count']
                 .median().reset_index())
order_profile.columns = ['month','dow','med_order_count']

# ── Returns profile — count by month ─────────────────────────────
df_returns['month'] = df_returns.return_date.dt.month
df_returns['year']  = df_returns.return_date.dt.year
ret_monthly   = df_returns.groupby(['year','month']).size().reset_index(name='return_count')
return_profile= (ret_monthly.groupby('month')['return_count']
                 .median().reset_index())
return_profile.columns = ['month','med_return_count']

# ── Promotions profile — promo count + avg discount by month ──────
promo_rows = []
for _, row in df_promos.iterrows():
    if pd.isna(row.end_date): continue
    for dt in pd.date_range(row.start_date, row.end_date, freq='D'):
        promo_rows.append({'date': dt, 'discount_value': row.discount_value})
df_promo_daily = pd.DataFrame(promo_rows)
df_promo_daily['month'] = df_promo_daily.date.dt.month
df_promo_daily['year']  = df_promo_daily.date.dt.year
promo_monthly = (df_promo_daily.groupby(['year','month'])
                 .agg(promo_count=('discount_value','count'),
                      avg_discount=('discount_value','mean'))
                 .reset_index())
promo_profile = (promo_monthly.groupby('month')
                 .agg(med_promo_count=('promo_count','median'),
                      med_avg_discount=('avg_discount','median'))
                 .reset_index())

print(f'Web profile  : {web_profile.shape}')
print(f'Order profile: {order_profile.shape}')
print(f'Return profile: {return_profile.shape}')
print(f'Promo profile: {promo_profile.shape}')


## 5. Historical Medians & Tet Multipliers

In [ ]:
# ── Historical median lookups (DOY, DOW, Month) ──────────────────
def compute_historical_medians(df_sales: pd.DataFrame) -> dict:
    df = df_sales.copy()
    df['doy']   = df.Date.dt.dayofyear
    df['dow']   = df.Date.dt.dayofweek
    df['month'] = df.Date.dt.month
    results = {}
    for col in ['Revenue', 'COGS']:
        results[f'doy_{col.lower()}']   = df.groupby('doy')[col].median()
        results[f'dow_{col.lower()}']   = df.groupby('dow')[col].median()
        results[f'month_{col.lower()}'] = df.groupby('month')[col].median()
    return results

hist_medians = compute_historical_medians(df_sales)
print('Historical medians computed:')
for k, v in hist_medians.items():
    print(f'  {k}: {len(v)} entries')


# ── Empirical Tet multipliers ──────────────────────────────────────
def compute_tet_multipliers(df_sales: pd.DataFrame, col: str) -> pd.Series:
    """median(sales[tet+delta] / yearly_median) for delta in [-30, +20]."""
    df = df_sales.set_index('Date')[[col]].copy()
    train_tets = [t for t in TET_DATES if df.index.min() <= t <= df.index.max()]
    multipliers = {}
    for delta in range(-30, 21):
        vals = []
        for tet_dt in train_tets:
            target_date = tet_dt + pd.Timedelta(days=delta)
            if target_date not in df.index: continue
            yr_data = df[df.index.year == target_date.year][col]
            if len(yr_data) < 30: continue
            yr_median = yr_data.median()
            if yr_median > 0:
                vals.append(df.loc[target_date, col] / yr_median)
        multipliers[delta] = np.median(vals) if vals else 1.0
    return pd.Series(multipliers, name=f'tet_mult_{col.lower()}')

tet_mult_rev  = compute_tet_multipliers(df_sales, 'Revenue')
tet_mult_cogs = compute_tet_multipliers(df_sales, 'COGS')
tet_mults_df  = pd.DataFrame({
    'delta':        tet_mult_rev.index,
    'revenue_mult': tet_mult_rev.values,
    'cogs_mult':    tet_mult_cogs.values,
})

# Visualise
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(tet_mult_rev.index,  tet_mult_rev.values,  marker='o', ms=4, label='Revenue', color='#2196F3')
ax.plot(tet_mult_cogs.index, tet_mult_cogs.values, marker='s', ms=4, label='COGS',    color='#FF9800')
ax.axhline(1.0, color='grey', ls='--', lw=0.8)
ax.axvline(0,   color='red',  ls='--', lw=1.2, label='Tet Day')
ax.set_xlabel('Days relative to Tet'); ax.set_ylabel('Multiplier')
ax.set_title('Empirical Tet Multipliers — Revenue & COGS')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Tet multipliers: delta range [{tet_mults_df.delta.min()}, {tet_mults_df.delta.max()}]')


## 6. Core Forecasting Utilities

In [ ]:
# ── Naive364 (DOW-preserving with fallback offsets) ──────────────
def naive364(df_sales: pd.DataFrame, forecast_dates: pd.DatetimeIndex, col: str) -> np.ndarray:
    history = df_sales.set_index('Date')[col]
    out = []
    for dt in forecast_dates:
        val = np.nan
        for offset in [364, 371, 357, 728]:
            cand = dt - pd.Timedelta(days=offset)
            if cand in history.index:
                val = history[cand]; break
        out.append(val if not np.isnan(val) else history.iloc[-1])
    return np.array(out)


# ── YoY growth factor ────────────────────────────────────────────
def yoy_growth(df_sales: pd.DataFrame, col: str, clip=(0.90, 1.15)) -> float:
    by_year = df_sales.groupby(df_sales.Date.dt.year)[col].sum()
    if len(by_year) >= 2:
        return float(np.clip(by_year.iloc[-1] / by_year.iloc[-2], *clip))
    return 1.0


# ── Apply Tet multipliers ────────────────────────────────────────
def apply_tet_mults(arr: np.ndarray, forecast_dates: pd.DatetimeIndex,
                    tet_mults_df: pd.DataFrame, col: str) -> np.ndarray:
    mult_col = 'revenue_mult' if col == 'Revenue' else 'cogs_mult'
    mult_map = dict(zip(tet_mults_df.delta.astype(int), tet_mults_df[mult_col]))
    out = arr.copy()
    for i, dt in enumerate(forecast_dates):
        for tet_dt in TET_DATES:
            delta = (dt - tet_dt).days
            if -30 <= delta <= 20 and delta in mult_map:
                out[i] = arr[i] * mult_map[delta]; break
    return out


# ── Feature matrix builder ───────────────────────────────────────
def build_feature_matrix(dates, df_cal, hist_medians,
                          web_profile, order_profile,
                          return_profile, promo_profile, col_suffix):
    feat = df_cal.reindex(pd.DatetimeIndex(dates)).copy()
    col  = col_suffix.lower()
    def _map(key):
        v = hist_medians.get(key, pd.Series())
        return v.squeeze('columns') if isinstance(v, pd.DataFrame) else v
    feat['hist_doy']   = feat['doy'].map(_map(f'doy_{col}'))
    feat['hist_dow']   = feat['dow'].map(_map(f'dow_{col}'))
    feat['hist_month'] = feat['month'].map(_map(f'month_{col}'))
    feat_r = feat.reset_index()
    feat_r = feat_r.merge(web_profile,    on=['month','dow'], how='left')
    feat_r = feat_r.merge(order_profile,  on=['month','dow'], how='left')
    feat_r = feat_r.merge(return_profile, on='month',         how='left')
    feat_r = feat_r.merge(promo_profile,  on='month',         how='left')
    feat_r = feat_r.set_index('ds')
    return feat_r


# ── LightGBM residual correction ─────────────────────────────────
def fit_lgbm_residual(df_train, X_train, naive_backbone, col, n_estimators=1000):
    actuals   = df_train.set_index('Date')[col].values
    residuals = actuals - naive_backbone
    n = len(df_train)
    sw = 1.0 + 2.0 * (np.arange(1, n+1)/n) ** 1.5  # recency weighting
    drop = ['date','year']
    X = X_train.drop(columns=[c for c in drop if c in X_train.columns], errors='ignore')
    X = X.select_dtypes(include=[np.number]).fillna(X.median())
    model = lgb.LGBMRegressor(
        n_estimators=n_estimators, learning_rate=0.02, num_leaves=31,
        min_child_samples=20, reg_alpha=0.5, reg_lambda=5.0,
        objective='mae', subsample=0.85, colsample_bytree=0.85,
        random_state=42, n_jobs=-1,
    )
    model.fit(X.values, residuals, sample_weight=sw)
    return model, X.columns.tolist()


# ── Scale correction ─────────────────────────────────────────────
def scale_to_target(pred: np.ndarray, target_mean: float) -> tuple:
    ratio = target_mean / max(np.mean(pred), 1e-8)
    return np.maximum(pred * ratio, 0), ratio


print('✅ Core utility functions defined.')


## 7. Prophet Feature Extractor + LightGBM Backbone (v3/v5)

In [ ]:
def fit_prophet_feature(df_sales, col, forecast_dates, df_holidays,
                        cps=0.05, smode='multiplicative', fourier=20,
                        start_year=None):
    """Fit Prophet and return train & forecast yhat arrays (as dicts keyed by date)."""
    df = df_sales[['Date', col]].rename(columns={'Date':'ds', col:'y'})
    if start_year:
        df = df[df.ds.dt.year >= start_year]
    cap   = df.y.quantile(0.995) * 1.25
    floor = max(df.y.quantile(0.005) * 0.75, 0)
    df['cap'] = cap; df['floor'] = floor

    m = Prophet(
        growth='logistic',
        yearly_seasonality=fourier,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode=smode,
        changepoint_prior_scale=cps,
        changepoint_range=0.9,
        holidays=df_holidays,
        holidays_prior_scale=10.0,
    )
    m.add_seasonality('monthly',   period=30.5,  fourier_order=5)
    m.add_seasonality('quarterly', period=91.25, fourier_order=3)
    m.fit(df)

    # Training predictions
    tr_fut = m.make_future_dataframe(periods=0, freq='D')
    tr_fut['cap'] = cap; tr_fut['floor'] = floor
    tr_pred = m.predict(tr_fut)
    train_map = dict(zip(tr_pred.ds, tr_pred.yhat))

    # Forecast predictions
    fc_df = pd.DataFrame({'ds': forecast_dates, 'cap': cap, 'floor': floor})
    fc_pred = m.predict(fc_df)
    fc_map = dict(zip(fc_pred.ds, fc_pred.yhat))

    return train_map, fc_map


print('✅ Prophet feature extractor ready.')


## 8. v5 — Naive-First Pipeline (Primary Backbone)

> **Critical insight:** Naive364 MAE ≈ 830k vs Prophet alone MAE ≈ 2.3M  
> Architecture: **Naive is backbone, LightGBM corrects Naive residuals**


In [ ]:
def weighted_median(values, weights):
    arr = np.array(values, dtype=float); wts = np.array(weights, dtype=float)
    idx = np.argsort(arr)
    arr_s, wts_s = arr[idx], wts[idx]
    cum = np.cumsum(wts_s)
    return arr_s[min(np.searchsorted(cum, cum[-1]/2), len(arr_s)-1)]


def multi_year_weighted_median(df_sales, forecast_dates, col):
    """Weighted median over same-DOW / ±3-DOY historical days. Recent years weighted higher."""
    df = df_sales.copy()
    df['dow'] = df.Date.dt.dayofweek
    df['doy'] = df.Date.dt.dayofyear
    df['year']= df.Date.dt.year
    min_yr, max_yr = df.year.min(), df.year.max()
    yr_range = max(max_yr - min_yr, 1)
    hist_idx = df_sales.set_index('Date')[col]

    out = []
    for dt in forecast_dates:
        target_dow, target_doy = dt.dayofweek, dt.dayofyear
        sub = df[df.year < dt.year].copy()
        doy_dist = sub.doy.apply(lambda d: min(abs(d-target_doy), 365-abs(d-target_doy)))
        work = sub[(sub.dow == target_dow) & (doy_dist <= 3)].copy()
        if len(work) == 0:
            for offset in [364, 371, 357, 728]:
                cand = dt - pd.Timedelta(days=offset)
                if cand in hist_idx.index:
                    out.append(hist_idx[cand]); break
            else:
                out.append(hist_idx.iloc[-1])
            continue
        work['weight'] = 1.0 + (work.year - min_yr) / yr_range
        out.append(weighted_median(work[col].values, work.weight.values))
    return np.array(out)


def build_v5_naive_backbone(df_sales, forecast_dates, col):
    """v5 backbone: 0.6×Naive364_tet + 0.4×MultiYearWeightedMedian."""
    n364     = naive364(df_sales, forecast_dates, col)
    g        = yoy_growth(df_sales, col, clip=(0.90, 1.15))
    n364_adj = apply_tet_mults(n364 * g, forecast_dates, tet_mults_df, col)
    mywm     = multi_year_weighted_median(df_sales, forecast_dates, col)
    return np.maximum(0.6 * n364_adj + 0.4 * mywm, 0)


print('✅ v5 Naive-First backbone ready.')


In [ ]:
# ── Generate v5 forecasts ──────────────────────────────────────────
v5_backbones    = {}
v5_corrections  = {}

for col in TARGET_COLS:
    print(f'\n=== v5 Building {col} ===')

    # Naive backbone on forecast
    backbone_fc = build_v5_naive_backbone(df_sales, forecast_dates, col)
    v5_backbones[col] = backbone_fc

    # Naive backbone approximation on training dates (for residual target)
    all_train_dates  = pd.DatetimeIndex(df_sales.Date.values)
    naive_train_approx = naive364(df_sales, all_train_dates, col)
    g = yoy_growth(df_sales, col, clip=(0.90, 1.15))
    naive_train_approx = naive_train_approx * g

    # Prophet as feature (minor role)
    print('  Fitting Prophet feature...')
    try:
        train_map, fc_map = fit_prophet_feature(df_sales, col, forecast_dates, df_holidays)
    except Exception as e:
        print(f'  Prophet failed: {e}')
        train_map = {}; fc_map = {}

    # Feature matrices
    X_train = build_feature_matrix(
        df_sales.Date.values, df_cal, hist_medians,
        web_profile, order_profile, return_profile, promo_profile, col
    )
    X_train['prophet_hint'] = pd.Series(train_map).reindex(X_train.index).values
    X_fc = build_feature_matrix(
        forecast_dates, df_cal, hist_medians,
        web_profile, order_profile, return_profile, promo_profile, col
    )
    X_fc['prophet_hint'] = pd.Series(fc_map).reindex(X_fc.index).values

    # Fit LightGBM on naive residuals
    lgbm_model, feat_cols = fit_lgbm_residual(
        df_sales, X_train, naive_train_approx, col, n_estimators=1200
    )

    X_fc_np   = X_fc[feat_cols].fillna(X_fc[feat_cols].median()).values
    correction = lgbm_model.predict(X_fc_np)

    # Damp correction at far horizons (exp decay)
    t = np.arange(N_FORECAST)
    damp = np.exp(-t / 300.0)
    v5_corrections[col] = correction * damp
    print(f'  Backbone mean: {backbone_fc.mean():,.0f}')
    print(f'  Correction mean (damped): {(correction*damp).mean():,.0f}')

# v5 variants
VARIANTS = {'corrected': 1.0, 'mild_correct': 0.5, 'pure_naive': 0.0}
v5_results = {}
for variant, strength in VARIANTS.items():
    v5_results[variant] = {
        col: np.maximum(v5_backbones[col] + strength * v5_corrections[col], 0)
        for col in TARGET_COLS
    }
    rv = v5_results[variant]['Revenue'].mean()
    cg = v5_results[variant]['COGS'].mean()
    print(f'v5_{variant:15s} | Revenue mean={rv:,.0f} | COGS mean={cg:,.0f}')


## 9. Theta Method (M-competition Winner)

In [ ]:
def theta_forecast(train_series: pd.Series, n_periods: int) -> np.ndarray:
    """
    Theta method:
    1. Deseasonalize (MA-7 + DOW seasonal factors)
    2. Theta1: OLS linear trend
    3. Theta2: SES alpha=0.2 (flat forecast)
    4. Blend 50/50 + re-seasonalize
    """
    y = train_series.values.copy().astype(float)
    n = len(y)
    dates = train_series.index

    # Deseasonalize
    ma7   = pd.Series(y).rolling(7, center=True, min_periods=1).mean().values
    ma7   = np.where(ma7 == 0, 1e-8, ma7)
    ratio = y / ma7
    dows  = pd.Series(dates.dayofweek)
    sf    = np.ones(7)
    for d in range(7):
        mask = dows == d
        if mask.sum() > 0:
            sf[d] = np.median(ratio[mask.values])
    sf = sf / sf.mean() if sf.mean() > 0 else sf
    deseas = y / sf[dows.values]

    # Theta1: OLS trend
    t_tr = np.arange(n).reshape(-1,1)
    t_fu = np.arange(n, n+n_periods).reshape(-1,1)
    theta1 = LinearRegression().fit(t_tr, deseas).predict(t_fu)

    # Theta2: SES
    alpha, sv = 0.2, deseas[0]
    for val in deseas:
        sv = alpha*val + (1-alpha)*sv
    theta2 = np.full(n_periods, sv)

    # Blend + re-seasonalize
    fut_dates = pd.date_range(train_series.index[-1] + pd.Timedelta(days=1),
                               periods=n_periods, freq='D')
    sf_fc = sf[fut_dates.dayofweek]
    return np.maximum((0.5*theta1 + 0.5*theta2) * sf_fc, 0)


# Test Theta
ts_2016 = df_sales[df_sales.Date.dt.year >= 2016].set_index('Date')['Revenue']
theta_test = theta_forecast(ts_2016, N_FORECAST)
print(f'Theta Revenue mean: {theta_test.mean():,.0f}')


## 10. v10/v11 — 10 Parallel Strategies

| Strategy | Description |
|----------|-------------|
| A | Naive 75% + v5_pure_naive 25% |
| B | Post-2018 training only |
| C | Blend best previous submissions |
| D | Sample submission as level anchor |
| E | **COGS = Revenue × historical ratio** (key insight) |
| F | Smoothed Tet calibration (Savitzky-Golay) |
| G | LightGBM + CatBoost residual ensemble |
| scaled | Scale-correct to match sample_submission level |
| statsfc | AutoTheta + AutoETS + SeasonalNaive (statsforecast) |
| post2019 | Post-COVID structural break models |


In [ ]:
# ── Strategy A: Naive 75% + Hybrid 25% ───────────────────────────
strategy_A = {}
for col in TARGET_COLS:
    n364 = apply_tet_mults(naive364(df_sales, forecast_dates, col) * yoy_growth(df_sales, col),
                           forecast_dates, tet_mults_df, col)
    hybrid = v5_results['pure_naive'][col]
    strategy_A[col] = np.maximum(0.75 * n364 + 0.25 * hybrid, 0)
    print(f'Strategy A | {col} mean: {strategy_A[col].mean():,.0f}')


In [ ]:
# ── Strategy B: Post-2018 Training ───────────────────────────────
strategy_B = {}
df_post2018 = df_sales[df_sales.Date.dt.year >= 2018].copy()
for col in TARGET_COLS:
    n364 = apply_tet_mults(
        naive364(df_post2018, forecast_dates, col) * yoy_growth(df_post2018, col, clip=(0.85,1.20)),
        forecast_dates, tet_mults_df, col
    )
    X_train_B = build_feature_matrix(
        df_post2018.Date.values, df_cal, hist_medians,
        web_profile, order_profile, return_profile, promo_profile, col
    )
    X_fc_B = build_feature_matrix(
        forecast_dates, df_cal, hist_medians,
        web_profile, order_profile, return_profile, promo_profile, col
    )
    naive_tr_B = naive364(df_post2018, pd.DatetimeIndex(df_post2018.Date.values), col)
    naive_tr_B *= yoy_growth(df_post2018, col, clip=(0.85,1.20))
    residuals_B = df_post2018.set_index('Date')[col].values - naive_tr_B
    m_B, fc_B = fit_lgbm_residual(df_post2018, X_train_B, naive_tr_B, col, n_estimators=1000)
    fc_B_cols = [c for c in fc_B if c in X_fc_B.columns]
    corr_B = m_B.predict(X_fc_B[fc_B_cols].fillna(X_fc_B[fc_B_cols].median()).values)
    damp_B = np.exp(-np.arange(N_FORECAST) / 200.0)
    strategy_B[col] = np.maximum(n364 + corr_B * damp_B, 0)
    print(f'Strategy B | {col} mean: {strategy_B[col].mean():,.0f}')


In [ ]:
# ── Strategy C: Blend Best Previous Submissions ───────────────────
strategy_C = {}
blend_weights = {'corrected': 0.4, 'mild_correct': 0.3, 'pure_naive': 0.2}
for col in TARGET_COLS:
    blend_pred = np.zeros(N_FORECAST); total_w = 0.0
    for variant, w in blend_weights.items():
        if variant in v5_results:
            blend_pred += w * v5_results[variant][col]; total_w += w
    strategy_C[col] = np.maximum(blend_pred / max(total_w, 1e-8), 0)
    print(f'Strategy C | {col} mean: {strategy_C[col].mean():,.0f}')


In [ ]:
# ── Strategy D: Sample Submission as Level Anchor ────────────────
strategy_D = {}
sample_indexed = df_sample.set_index('Date')
for col in TARGET_COLS:
    model_pred  = v5_results['corrected'][col]
    sample_pred = sample_indexed[col].reindex(forecast_dates).values
    scale       = np.nanmean(sample_pred) / max(np.nanmean(model_pred), 1e-8)
    model_scaled= model_pred * scale
    pred_D      = np.where(np.isnan(sample_pred), model_scaled,
                           0.6 * model_scaled + 0.4 * sample_pred)
    strategy_D[col] = np.maximum(pred_D, 0)
    print(f'Strategy D | {col} mean: {strategy_D[col].mean():,.0f} (scale={scale:.3f})')


In [ ]:
# ── Strategy E: COGS = Revenue × Historical Ratio (KEY INSIGHT) ───
df_sales['cogs_ratio'] = df_sales.COGS / df_sales.Revenue
ratio_by_month = (
    df_sales[df_sales.Date.dt.year >= 2019]
    .groupby(df_sales.Date.dt.month).cogs_ratio.median()
)
rev_base      = v5_results['corrected']['Revenue']
fc_months     = forecast_dates.month
ratio_monthly = np.array([ratio_by_month.get(m, ratio_by_month.mean()) for m in fc_months])

strategy_E = {
    'Revenue': np.maximum(rev_base, 0),
    'COGS':    np.maximum(rev_base * ratio_monthly, 0),
}
print(f'Strategy E | Revenue mean: {strategy_E["Revenue"].mean():,.0f}')
print(f'Strategy E | COGS mean   : {strategy_E["COGS"].mean():,.0f}')
print(f'Implied COGS/Rev ratio   : {(strategy_E["COGS"]/strategy_E["Revenue"]).mean():.4f}')
print('\n📊 COGS/Revenue ratio stability (2019+):')
print(ratio_by_month.round(4))


In [ ]:
# ── Strategy F: Smoothed Tet Calibration ─────────────────────────
def compute_smoothed_tet_mults(df_sales, col, smooth_window=5):
    df = df_sales.set_index('Date')[[col]].copy()
    train_tets = [t for t in TET_DATES if df.index.min() <= t <= df.index.max()]
    raw = {}
    for delta in range(-30, 21):
        vals = []
        for tet_dt in train_tets:
            td = tet_dt + pd.Timedelta(days=delta)
            if td not in df.index: continue
            yr_data = df[df.index.year == td.year][col]
            if len(yr_data) < 30: continue
            ym = yr_data.median()
            if ym > 0: vals.append(df.loc[td, col] / ym)
        raw[delta] = np.median(vals) if vals else 1.0
    deltas = np.array(sorted(raw.keys()))
    mults  = np.array([raw[d] for d in deltas])
    if smooth_window > 0 and len(mults) >= smooth_window:
        mults = savgol_filter(mults, window_length=min(smooth_window, len(mults)//2*2+1), polyorder=2)
    return dict(zip(deltas, mults))

strategy_F = {}
for col in TARGET_COLS:
    smooth_map = compute_smoothed_tet_mults(df_sales, col, smooth_window=5)
    base = v5_results['corrected'][col].copy()
    for i, dt in enumerate(forecast_dates):
        for tet_dt in TET_DATES:
            delta = (dt - tet_dt).days
            if -30 <= delta <= 20 and delta in smooth_map:
                base[i] = v5_results['corrected'][col][i] * smooth_map[delta]; break
    strategy_F[col] = np.maximum(base, 0)
    print(f'Strategy F | {col} mean: {strategy_F[col].mean():,.0f}')


In [ ]:
# ── Strategy G: LightGBM + CatBoost Ensemble ─────────────────────
strategy_G = {}
if HAS_CATBOOST:
    for col in TARGET_COLS:
        print(f'Fitting CatBoost for {col}...')
        n364_g = apply_tet_mults(naive364(df_sales, forecast_dates, col) * yoy_growth(df_sales, col),
                                  forecast_dates, tet_mults_df, col)
        X_train_G = build_feature_matrix(df_sales.Date.values, df_cal, hist_medians,
                                          web_profile, order_profile, return_profile, promo_profile, col)
        X_fc_G    = build_feature_matrix(forecast_dates, df_cal, hist_medians,
                                          web_profile, order_profile, return_profile, promo_profile, col)
        naive_tr_G = naive364(df_sales, pd.DatetimeIndex(df_sales.Date.values), col) * yoy_growth(df_sales, col)
        residuals_G= df_sales.set_index('Date')[col].values - naive_tr_G
        drop = ['date','year']
        X_g  = X_train_G.drop(columns=[c for c in drop if c in X_train_G.columns], errors='ignore')
        X_g  = X_g.select_dtypes(include=[np.number]).fillna(X_g.median())
        fc_g = X_fc_G.reindex(columns=X_g.columns).fillna(X_fc_G.median())
        n    = len(df_sales)
        sw   = 1.0 + 2.0 * (np.arange(1,n+1)/n)**1.5
        m_lgb = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.02, num_leaves=31,
                                    objective='mae', subsample=0.85, colsample_bytree=0.85,
                                    random_state=42, n_jobs=-1)
        m_lgb.fit(X_g.values, residuals_G, sample_weight=sw)
        m_cat = CatBoostRegressor(iterations=800, learning_rate=0.03, depth=6,
                                   loss_function='MAE', random_seed=42, verbose=False)
        m_cat.fit(X_g.values, residuals_G, sample_weight=sw)
        lgb_c = m_lgb.predict(fc_g.values)
        cat_c = m_cat.predict(fc_g.values)
        damp  = np.exp(-np.arange(N_FORECAST) / 250.0)
        ensemble_c = (0.6 * lgb_c + 0.4 * cat_c) * damp
        strategy_G[col] = np.maximum(n364_g + ensemble_c, 0)
        print(f'  {col} mean: {strategy_G[col].mean():,.0f}')
else:
    print('⚠️  CatBoost not available. Using strategy_A as fallback for G.')
    strategy_G = {col: strategy_A[col].copy() for col in TARGET_COLS}


In [ ]:
# ── statsforecast: AutoTheta + AutoETS + SeasonalNaive ───────────
statsfc_results = {}
if HAS_STATSFORECAST:
    for col in TARGET_COLS:
        print(f'\nFitting statsforecast for {col}...')
        df_p19 = df_sales[df_sales.Date.dt.year >= 2019].copy()
        sf_df  = pd.DataFrame({'unique_id': col, 'ds': df_p19.Date, 'y': df_p19[col].values})
        models_sf = [AutoTheta(season_length=7), SeasonalNaive(season_length=7)]
        try:    models_sf.append(AutoETS(season_length=7))
        except: pass
        sf       = StatsForecast(models=models_sf, freq='D', n_jobs=1)
        sf_pred  = sf.forecast(df=sf_df, h=N_FORECAST)
        mc       = [c for c in sf_pred.columns if c not in ['unique_id','ds']]
        statsfc_results[col] = np.maximum(sf_pred[mc].mean(axis=1).values, 0)
        print(f'  statsforecast {col} mean: {statsfc_results[col].mean():,.0f}')
else:
    print('⚠️  Using Naive364 as statsforecast fallback.')
    for col in TARGET_COLS:
        df_p19 = df_sales[df_sales.Date.dt.year >= 2019]
        statsfc_results[col] = np.maximum(
            naive364(df_p19, forecast_dates, col) * yoy_growth(df_p19, col), 0)

# ── Scale correction — match to sample_submission level ──────────
scaled_results = {}
scale_factors  = {}
for col in TARGET_COLS:
    pred_sc, ratio = scale_to_target(strategy_A[col], df_sample[col].mean())
    scaled_results[col] = pred_sc
    scale_factors[col]  = ratio
    print(f'Scaled {col}: factor={ratio:.4f} | mean after={pred_sc.mean():,.0f}')

# ── Post-2019 COVID-break models ──────────────────────────────────
post2019_results = {}
df_post2019 = df_sales[df_sales.Date.dt.year >= 2019].copy()
for col in TARGET_COLS:
    n364_p19     = apply_tet_mults(
        naive364(df_post2019, forecast_dates, col) * yoy_growth(df_post2019, col, clip=(0.90,1.15)),
        forecast_dates, tet_mults_df, col
    )
    combined_p19 = (0.5*n364_p19 + 0.5*statsfc_results[col]) if statsfc_results else n364_p19
    post2019_results[col] = np.maximum(combined_p19, 0)
    print(f'Post-2019 | {col} mean: {post2019_results[col].mean():,.0f}')


## 11. Final Weighted Ensemble — All 10 Strategies

In [ ]:
ENSEMBLE_WEIGHTS = {
    'A':        0.10,  # Naive 75% + v5 25%
    'B':        0.10,  # Post-2018
    'C':        0.10,  # Blend best prev
    'D':        0.10,  # Sample sub anchor
    'E':        0.15,  # COGS=Rev×ratio (strategic)
    'F':        0.10,  # Smoothed Tet
    'G':        0.05,  # CatBoost ensemble
    'scaled':   0.10,  # Scale-corrected
    'statsfc':  0.10,  # M-competition models
    'post2019': 0.10,  # COVID-break aware
}
assert abs(sum(ENSEMBLE_WEIGHTS.values()) - 1.0) < 1e-6, 'Weights must sum to 1'

strategies_map = {
    'A': strategy_A, 'B': strategy_B, 'C': strategy_C,
    'D': strategy_D, 'E': strategy_E, 'F': strategy_F,
    'G': strategy_G,
    'scaled':   {col: scaled_results[col]   for col in TARGET_COLS},
    'statsfc':  {col: statsfc_results[col]  for col in TARGET_COLS},
    'post2019': {col: post2019_results[col] for col in TARGET_COLS},
}

final_results = {}
for col in TARGET_COLS:
    pred = np.zeros(N_FORECAST)
    for key, w in ENSEMBLE_WEIGHTS.items():
        pred += w * strategies_map[key][col]
    final_results[col] = np.maximum(pred, 0)
    print(f'Final Ensemble | {col} mean: {final_results[col].mean():,.0f}')

# Variant: COGS derived from Revenue × stable ratio (reduces independent COGS error)
ratio_recent = (
    df_sales[df_sales.Date.dt.year >= 2019]
    .groupby(df_sales.Date.dt.month)
    .apply(lambda x: (x.COGS / x.Revenue).median())
)
ratio_arr = np.array([ratio_recent.get(m, ratio_recent.mean()) for m in forecast_dates.month])

final_e_variant = {
    'Revenue': final_results['Revenue'],
    'COGS':    np.maximum(final_results['Revenue'] * ratio_arr, 0),
}
print(f'\nFinal E-variant | Revenue mean: {final_e_variant["Revenue"].mean():,.0f}')
print(f'Final E-variant | COGS mean   : {final_e_variant["COGS"].mean():,.0f}')
print(f'Implied ratio   : {(final_e_variant["COGS"]/final_e_variant["Revenue"]).mean():.4f}')


## 12. Forecast Comparison Dashboard

In [ ]:
sample_rev = df_sample.Revenue.mean()
sample_cog = df_sample.COGS.mean()

all_subs = [
    ('v5_pure_naive',       v5_results['pure_naive']),
    ('v5_mild_correct',     v5_results['mild_correct']),
    ('v5_corrected',        v5_results['corrected']),
    ('v10_E_cogs_ratio',    strategy_E),
    ('v10_D_sample_anchor', strategy_D),
    ('v11_scaled',          {col: scaled_results[col] for col in TARGET_COLS}),
    ('v11_final_ensemble',  final_results),
    ('v11_final_e_variant', final_e_variant),
]

print('=' * 75)
print(f'{"Submission":<30} {"Revenue Mean":>15} {"COGS Mean":>15} {"Rev/Sample":>10}')
print('=' * 75)
print(f'{"SAMPLE_SUBMISSION (ref)":<30} {sample_rev:>15,.0f} {sample_cog:>15,.0f} {1.0:>10.2f}x')
print('-' * 75)
for fname, pred_dict in all_subs:
    rv = np.maximum(pred_dict['Revenue'], 0).mean()
    cg = np.maximum(pred_dict['COGS'],    0).mean()
    print(f'{fname:<30} {rv:>15,.0f} {cg:>15,.0f} {rv/sample_rev:>10.2f}x')
print('=' * 75)


In [ ]:
# ── Plot comparison ────────────────────────────────────────────────
colors_map = {
    'v5_pure_naive':       '#9E9E9E',
    'v11_scaled':          '#2196F3',
    'v10_D_sample_anchor': '#4CAF50',
    'v10_E_cogs_ratio':    '#FF5722',
    'v11_final_ensemble':  '#E91E63',
    'v11_final_e_variant': '#9C27B0',
}

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
for ax_idx, col in enumerate(['Revenue', 'COGS']):
    ax = axes[ax_idx]
    # Historical actuals (last 2 years)
    hist = df_sales[df_sales.Date.dt.year >= 2020]
    ax.plot(hist.Date, hist[col], color='#607D8B', lw=0.6, alpha=0.5, label='Historical')
    # Sample submission as reference
    ax.plot(df_sample.Date, df_sample[col], color='black', lw=1.5, ls='--',
            alpha=0.8, label='Sample Sub', zorder=10)
    # Forecast submissions
    for fname, pred_dict in all_subs:
        if fname in colors_map:
            ax.plot(forecast_dates, np.maximum(pred_dict[col], 0),
                    color=colors_map[fname], lw=1.0, alpha=0.85, label=fname)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    ax.set_ylabel(col); ax.legend(loc='upper right', fontsize=7)

axes[0].set_title('Forecast Comparison — All v10/v11 Strategies', fontsize=14, fontweight='bold')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout(); plt.show()


In [ ]:
# ── Revenue & COGS trend plot with Tet markers ────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)
for ax, col, color in zip(axes, ['Revenue','COGS'], ['#2196F3','#FF9800']):
    ax.plot(df_sales.Date, df_sales[col], lw=0.8, color=color, alpha=0.8)
    ax.fill_between(df_sales.Date, df_sales[col], alpha=0.1, color=color)
    for tet_dt in TET_DATES:
        if df_sales.Date.min() <= tet_dt <= df_sales.Date.max():
            ax.axvline(tet_dt, color='red', lw=0.5, alpha=0.4)
    ax.set_ylabel(col)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[0].set_title('Historical Revenue & COGS (red lines = Tet)', fontweight='bold')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()


## 13. Save All Submissions

In [ ]:
to_save = [
    ('v5_pure_naive',       v5_results['pure_naive']),
    ('v5_mild_correct',     v5_results['mild_correct']),
    ('v5_corrected',        v5_results['corrected']),
    ('v10_A_naive75',       strategy_A),
    ('v10_B_post2018',      strategy_B),
    ('v10_C_blend',         strategy_C),
    ('v10_D_sample_anchor', strategy_D),
    ('v10_E_cogs_ratio',    strategy_E),
    ('v10_F_tet_smooth',    strategy_F),
    ('v10_G_catboost',      strategy_G),
    ('v11_scaled',          {col: scaled_results[col]   for col in TARGET_COLS}),
    ('v11_statsforecast',   {col: statsfc_results[col]  for col in TARGET_COLS}),
    ('v11_post2019',        {col: post2019_results[col] for col in TARGET_COLS}),
    ('v11_final_ensemble',  final_results),
    ('v11_final_e_variant', final_e_variant),
]

for fname, pred_dict in to_save:
    sub = pd.DataFrame({
        'Date':    forecast_dates.strftime('%Y-%m-%d'),
        'Revenue': np.maximum(pred_dict['Revenue'], 0),
        'COGS':    np.maximum(pred_dict['COGS'],    0),
    })
    assert len(sub) == N_FORECAST, f'{fname}: expected {N_FORECAST} rows'
    assert (sub.Revenue > 0).all(), f'{fname}: non-positive Revenue'
    assert (sub.COGS > 0).all(),    f'{fname}: non-positive COGS'
    sub.to_csv(OUTPUT_DIR / f'{fname}.csv', index=False)
    print(f'✅ Saved {fname:30s} | Rev={sub.Revenue.mean():,.0f} | COGS={sub.COGS.mean():,.0f}')

print(f'\n📁 All submissions saved to: {OUTPUT_DIR}')


## 14. Submission Recommendation

In [ ]:
print('=' * 65)
print('SUBMISSION RECOMMENDATIONS')
print('=' * 65)
print()
print('🥇 PRIMARY   → v11_final_ensemble.csv')
print('   Blends all 10 strategies with higher weight on scale-')
print('   aware and post-COVID models. Most robust.')
print()
print('🥈 BACKUP-1  → v11_final_e_variant.csv')
print('   COGS derived from Revenue × stable COGS/Rev ratio.')
print('   Avoids independent COGS error propagation.')
print()
print('🥉 BACKUP-2  → v10_E_cogs_ratio.csv')
print('   Purest COGS-ratio insight. Simple and interpretable.')
print()
print('🔬 DIAGNOSTIC→ v11_scaled.csv')
print('   Force-scaled to sample_sub level. Diagnose scale gap.')
print()

# Key insight
median_ratio = df_sales[df_sales.Date.dt.year >= 2019].apply(
    lambda r: r.COGS/r.Revenue if r.Revenue > 0 else np.nan, axis=1).median()
print(f'📊 Strategy E key insight:')
print(f'   COGS/Revenue ratio (2019-2022): ~{median_ratio:.4f}')
print(f'   Sample sub Revenue mean: {df_sample.Revenue.mean():,.0f}')
print(f'   Sample sub COGS mean  : {df_sample.COGS.mean():,.0f}')
print()
print('✅ All forecasts complete!')
print(f'Files saved to: {OUTPUT_DIR}')
